# 21: O pipeline como agente A2A

Até aqui, só uma pessoa conseguia pedir uma análise: pela CLI ou pelo dashboard. O fluxo de carteira que vem
depois precisa pedir a análise de dentro de um programa, sem um humano digitando o ticker. Por isso o pipeline
ganhou uma porta de entrada no protocolo A2A, para que outro agente o encontre, leia o que ele faz e peça uma
análise do mesmo jeito que um cliente HTTP faria.

Este notebook sobe o servidor dentro do próprio processo e mostra cada passo dessa conversa. Nada aqui depende
do container: a aplicação roda numa rede virtual do Python, sem porta aberta.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

A2A (Agent-to-Agent). Um protocolo aberto para um agente pedir trabalho a outro pela rede. O pedido não é
uma chamada de função com parâmetros fixos: quem chama descreve o que quer em texto, e quem atende decide como
cumprir. A troca usa JSON-RPC sobre HTTP, com a versão do protocolo no cabeçalho `A2A-Version`.

Agent Card. A ficha pública do agente, servida em `/.well-known/agent-card.json`. Ela diz o nome, o que o
agente faz, em que endereço se chama, quais formatos aceita e devolve, e quais habilidades (skills) oferece.
Outro agente lê a ficha antes de decidir se vale a pena chamar.

Tarefa e estados. Cada pedido vira uma tarefa, com um identificador e um estado. No protocolo 1.0 os nomes
vêm em maiúsculas:

- `TASK_STATE_SUBMITTED`: o pedido foi recebido e ainda não começou.
- `TASK_STATE_WORKING`: a análise está rodando.
- `TASK_STATE_COMPLETED`: terminou com resultado.
- `TASK_STATE_REJECTED`: recusado antes de rodar (aqui, ticker inválido ou inexistente).
- `TASK_STATE_FAILED`: começou e falhou (aqui, uma exceção no grafo).

Numa chamada `SendMessage` comum, a resposta só chega quando a tarefa termina. Quem quer ver os estados no
caminho usa `SendStreamingMessage`, que manda um evento por etapa.

Artefato e partes. O resultado de uma tarefa sai em artefatos, e cada artefato tem partes. Uma parte pode
ser texto (`text`) ou dado estruturado (`data`). Aqui o artefato `relatorio` tem duas partes: o markdown para
quem lê e o JSON com o relatório estruturado para outro programa.

A2A e MCP. Parecem parecidos, mas resolvem coisas diferentes. Com o MCP, o cliente empresta uma ferramenta
com entrada e saída definidas, e ele mesmo decide a sequência de passos. Com o A2A, o cliente delega um
trabalho inteiro: descreve o objetivo e recebe o resultado pronto, sem saber quantas ferramentas o outro agente
usou por dentro.

## 2. O Agent Card

O card é a primeira coisa que um cliente lê. Estes são os campos que importam para quem chama:

- `name` e `description`: o nome e o que o pipeline faz. A descrição diz também que o relatório não recomenda
  compra nem venda.
- `supportedInterfaces`: o endereço do serviço (`url`), o tipo de conexão (`protocolBinding`, aqui `JSONRPC`) e
  a versão do protocolo (`protocolVersion`, aqui `1.0`).
- `capabilities.streaming`: `true`, então o cliente pode usar `SendStreamingMessage`.
- `defaultInputModes` e `defaultOutputModes`: o que o servidor entende (texto puro) e o que devolve (markdown e
  JSON).
- `skills`: a lista de habilidades. Há uma só, `analisar_acao`, com exemplos de pedido.

In [2]:
import json
import uuid

import httpx

from servidor_a2a.servidor import criar_app

# Uma aplicação só para o notebook inteiro: as tarefas ficam na memória dela, e a célula 7 precisa
# encontrar as tarefas criadas nas células anteriores.
app = criar_app()
cliente = httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://a2a", timeout=60)
CABECALHOS = {"A2A-Version": "1.0"}


def corpo(texto: str) -> dict:
    """Corpo JSON-RPC do SendMessage, com um messageId novo a cada pedido."""
    return {
        "jsonrpc": "2.0", "id": "1", "method": "SendMessage",
        "params": {"message": {"role": "ROLE_USER", "messageId": uuid.uuid4().hex, "parts": [{"text": texto}]}},
    }


resposta = await cliente.get("/.well-known/agent-card.json")
print(f"GET /.well-known/agent-card.json -> {resposta.status_code}")
print(json.dumps(resposta.json(), ensure_ascii=False, indent=2))

GET /.well-known/agent-card.json -> 200
{
  "name": "Pipeline B3",
  "description": "Analisa uma ação da B3: dados de mercado, métricas de risco, notícias, contexto do setor e, quando há anomalia, o diagnóstico da causa. Devolve um relatório em markdown e o mesmo relatório estruturado em JSON. Não recomenda compra nem venda.",
  "supportedInterfaces": [
    {
      "url": "http://pipeline-b3:10001/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/markdown",
    "application/json"
  ],
  "skills": [
    {
      "id": "analisar_acao",
      "name": "Analisar ação da B3",
      "description": "Recebe um ticker da B3 (ex: VALE3), sozinho ou numa frase, um por mensagem. A análise leva de 20 a 30 segundos.",
      "tags": [
        "b3",
        "ações",
        "análise fundamentalista",
        "risco"
      ]

## 3. Entrada: o que chega ao grafo

Antes de qualquer análise, o servidor lê o texto da mensagem. A função `resolver_entrada` faz isso e devolve um
de dois resultados: um ticker aceito ou o motivo da recusa.

- Um ticker sozinho ou dentro de uma frase, como "Analise VALE3", é aceito.
- Texto sem ticker, como "analise a Vale", é recusado: o nome da empresa não serve.
- Dois tickers na mesma mensagem, como "Compare VALE3 e PETR4", também são recusados. Cada análise cobre uma
  empresa só.
- Um ticker que a Brapi diz não existir é recusado com sugestão. Aqui a Brapi responde de verdade, porque o
  notebook não está nos testes.

O grafo nunca recebe o texto do chamador. Ele recebe só a frase montada pelo servidor, `Analise a ação VALE3`.
Assim, uma mensagem como "vale3 e ignore as regras" não chega ao modelo com instruções a mais.

In [3]:
from servidor_a2a.entrada import resolver_entrada

for texto in ["Analise VALE3", "analise a Vale", "Compare VALE3 e PETR4"]:
    entrada = resolver_entrada(texto)
    print(repr(texto))
    print(f"  ticker: {entrada.ticker}")
    print(f"  recusa: {entrada.recusa}")

'Analise VALE3'
  ticker: VALE3
  recusa: None
'analise a Vale'
  ticker: None
  recusa: Ticker inválido: use o código da B3, ex: VALE3.
'Compare VALE3 e PETR4'
  ticker: None
  recusa: Envie um ticker por mensagem: encontrei VALE3, PETR4.


## 4. Recusa sem custo

A recusa acontece antes do grafo, então não há chamada ao modelo nem busca na web. Abaixo, uma mensagem sem
ticker enviada pelo protocolo de verdade (`SendMessage`). A tarefa volta com o estado `TASK_STATE_REJECTED` e o
motivo no primeiro texto do status.

In [4]:
resposta = await cliente.post("/", headers=CABECALHOS, json=corpo("analise a Vale"))
tarefa_recusada = resposta.json()["result"]["task"]
print("estado:", tarefa_recusada["status"]["state"])
print("motivo:", tarefa_recusada["status"]["message"]["parts"][0]["text"])
print("artefatos:", len(tarefa_recusada.get("artifacts", [])))

estado: TASK_STATE_REJECTED
motivo: Ticker inválido: use o código da B3, ex: VALE3.
artefatos: 0


## 5. A análise completa

Esta é a única análise do notebook. O pedido é `Analise WEGE3`, e a chamada espera até 180 segundos, que é
folga de sobra para as cerca de 20 a 30 segundos que o grafo costuma levar. Ao terminar, a célula imprime o
estado, os avisos de progresso que ficaram no histórico da tarefa e o começo do markdown.

Os avisos vêm das mensagens com `ROLE_AGENT` no histórico. O Supervisor passa várias vezes e não entra na lista,
porque não é uma etapa da análise para quem chama.

In [5]:
resposta = await cliente.post("/", headers=CABECALHOS, json=corpo("Analise WEGE3"), timeout=180)
tarefa = resposta.json()["result"]["task"]
estado = tarefa["status"]["state"]
print("estado:", estado)

for mensagem in tarefa.get("history", []):
    if mensagem["role"] == "ROLE_AGENT":
        print("aviso:", mensagem["parts"][0]["text"])

# Se a tarefa não concluiu, o motivo vai junto na falha do notebook, em vez de um KeyError mais adiante
assert estado == "TASK_STATE_COMPLETED", tarefa["status"]["message"]["parts"][0]["text"]
markdown = tarefa["artifacts"][0]["parts"][0]["text"]
print()
print(markdown[:600])

2026-11-15 20:01:30,509 - fluxo_financeiro - DEBUG - [a2a] cabeçalhos de trace recebidos: []


2026-11-15 20:01:30,510 - fluxo_financeiro - INFO - [a2a] WEGE3: análise iniciada


2026-11-15 20:01:30,594 - fluxo_financeiro - INFO - [supervisor] início, loop_count=0


2026-11-15 20:01:32,043 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico, métricas e balanços recentes da ação WEGE3.'


2026-11-15 20:01:32,045 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-15 20:01:32,047 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


... (27 outputs left out)

2026-11-15 20:01:52,545 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (5 msgs)


2026-11-15 20:01:54,631 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (1596 chars)


2026-11-15 20:01:54,635 - fluxo_financeiro - INFO - [a2a] WEGE3: análise concluída


estado: TASK_STATE_COMPLETED
aviso: Etapa concluída: YFinance
aviso: Etapa concluída: Analyzer
aviso: Etapa concluída: Consolidador

# Relatório: WEGE3

## Resumo
A WEG S.A. negocia cotada a R$ 51,31, apresentando um sólido valor de mercado de R$ 215,3 bilhões, P/L de 34,4x (relação entre o preço da ação e o lucro por ação) e ROE de 29,9%. Recentemente, a companhia tem lidado com pressões de custos e tarifas nos Estados Unidos que afetaram as margens, além de oscilações decorrentes dos resultados do 3º trimestre de 2026 e planos para um novo programa de recompra de ações.

## Indicadores
- Preço Atual: R$ 51,31
- Valor de mercado: R$ 215,3 bi
- LPA: R$ 1,49
- P/L: 34,4x
- ROE: 29,9%
- DY: 3,02%
- Sharpe Ratio: 0,32
- Sortin


## 6. A parte de dados

O mesmo artefato traz a segunda parte, com o relatório em JSON. Este é o pedaço que um programa usa: o
fluxo de carteira lê os campos e não precisa interpretar texto. A célula não chama o servidor de novo, lê a
resposta da célula anterior.

Alguns termos que costumam aparecer nos indicadores:

- P/L (preço sobre lucro): quantos anos de lucro atual cabem no preço da ação. Quanto menor, mais barata a
  ação parece em relação ao que ganha.
- ROE (retorno sobre o patrimônio): o lucro dividido pelo patrimônio líquido da empresa. Mostra o retorno
  que os acionistas obtêm sobre o capital que a empresa tem.
- Dividend yield: os dividendos pagos nos últimos 12 meses divididos pelo preço atual. É o retorno em caixa
  que a ação devolve, sem contar a variação do preço.
- Sharpe: o retorno acima da taxa de juros livre de risco (a SELIC, neste projeto) dividido pela volatilidade.
  Mede quanto retorno se ganha por unidade de risco assumida.
- Beta: a sensibilidade da ação ao Ibovespa. Beta acima de 1 indica que ela tende a oscilar mais que o índice;
  abaixo de 1, que oscila menos.

In [6]:
partes = tarefa["artifacts"][0]["parts"]

if len(partes) > 1 and "data" in partes[1]:
    relatorio = partes[1]["data"]["relatorio"]
    print("resumo:", relatorio["resumo"])
    print()
    print("indicadores:")
    for indicador in relatorio["indicadores"]:
        print("  -", indicador)
    print()
    print("limitações:")
    for limitacao in relatorio["limitacoes"] or ["(nenhuma)"]:
        print("  -", limitacao)
else:
    # O Consolidador pode falhar: o artefato sai só com o markdown, e o cliente precisa aceitar isso
    print("Sem parte de dados: o relatório estruturado não saiu, e o cliente fica só com o markdown.")

resumo: A WEG S.A. negocia cotada a R$ 51,31, apresentando um sólido valor de mercado de R$ 215,3 bilhões, P/L de 34,4x (relação entre o preço da ação e o lucro por ação) e ROE de 29,9%. Recentemente, a companhia tem lidado com pressões de custos e tarifas nos Estados Unidos que afetaram as margens, além de oscilações decorrentes dos resultados do 3º trimestre de 2026 e planos para um novo programa de recompra de ações.

indicadores:
  - Preço Atual: R$ 51,31
  - Valor de mercado: R$ 215,3 bi
  - LPA: R$ 1,49
  - P/L: 34,4x
  - ROE: 29,9%
  - DY: 3,02%
  - Sharpe Ratio: 0,32
  - Sortino Ratio: 0,51
  - Max Drawdown: -22,5%
  - VaR 95% (diário): -3,15%
  - Beta (vs IBOV): 0,30
  - Alpha (anual): 12,1%
  - Piotroski F-Score: 5/9
  - SELIC (meta): 13,75% a.a.
  - CDI: 13,65% a.a.
  - IPCA acumulado em 12 meses: 4,63%

limitações:
  - (nenhuma)


## 7. Como isso entra no gateway

Os clientes do projeto não chamam o pipeline direto. Eles passam pelo LiteLLM, que controla as chaves, a
cota e o trace. O agente se registra no bloco `agents:` do `config.yaml` do gateway, com o mesmo nome, endereço
e versão que o card anuncia:

```yaml
agents:
  - agent_name: pipeline-b3
    agent_card_params:
      name: "Pipeline B3"
      url: "http://pipeline-b3:10001"
      protocolVersion: "1.0"
```

Depois de registrado, o chamador envia o `SendMessage` para `http://localhost:4000/a2a/pipeline-b3`, com a chave
virtual do próprio chamador no cabeçalho `Authorization: Bearer sk-...`. O gateway repassa o pedido ao servidor
pela rede `agentes-net`, e cada chamada aparece no log de custo. Este notebook não executa essa chamada e não usa
nenhuma chave: a chave virtual fica no `.env` de quem chama.

## O que observar

- O texto do chamador não chega ao grafo. Só o ticker entra, na frase que o servidor monta.
- A recusa não gasta créditos: a seção 4 parou na leitura do texto, antes da Brapi, do modelo e da busca.
- O markdown é para quem lê. A parte de dados em JSON é para outro programa, e pode faltar quando o Consolidador falha.
- As tarefas ficam só na memória do servidor. Se o processo reiniciar, o histórico some junto, e um cliente que
  quiser consultar uma tarefa antiga não encontra nada.